In [1]:

!pip install torchmeta==1.4.0


In [2]:
import torch
import torch.nn as nn

from collections import OrderedDict

class MetaModule(nn.Module):
    """
    Base class for PyTorch meta-learning modules. These modules accept an
    additional argument `params` in their `forward` method.

    Notes
    -----
    Objects inherited from `MetaModule` are fully compatible with PyTorch
    modules from `torch.nn.Module`. The argument `params` is a dictionary of
    tensors, with full support of the computation graph (for differentiation).
    """
    def meta_named_parameters(self, prefix='', recurse=True):
        gen = self._named_members(
            lambda module: module._parameters.items()
            if isinstance(module, MetaModule) else [],
            prefix=prefix, recurse=recurse)
        for elem in gen:
            yield elem

    def meta_parameters(self, recurse=True):
        for name, param in self.meta_named_parameters(recurse=recurse):
            yield param

In [3]:
import math
import torch.nn as nn
import torch.nn.functional as F

from collections import OrderedDict

class MetaLinear(nn.Linear, MetaModule):
    __doc__ = nn.Linear.__doc__

    def reset_parameters(self):
        # nn.init.uniform_(self.weight, -0.1, 0.1)
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)
    
    def forward(self, input, params=None):
        if params is None:
            params = OrderedDict(self.named_parameters())
        bias = params.get('bias', None)
        return F.linear(input, params['weight'], bias)

class MetaBilinear(nn.Bilinear, MetaModule):
    __doc__ = nn.Bilinear.__doc__

    def forward(self, input1, input2, params=None):
        if params is None:
            params = OrderedDict(self.named_parameters())
        bias = params.get('bias', None)
        return F.bilinear(input1, input2, params['weight'], bias)

In [4]:
import re
from collections import OrderedDict

def get_subdict(dictionary, key=None):
    if dictionary is None:
        return None
    if (key is None) or (key == ''):
        return dictionary
    key_re = re.compile(r'^{0}\.(.+)'.format(re.escape(key)))
    return OrderedDict((key_re.sub(r'\1', k), value) for (k, value)
        in dictionary.items() if key_re.match(k) is not None)

In [5]:
import torch.nn as nn
import torch.nn.functional as F

from collections import OrderedDict
from torch.nn.modules.batchnorm import _BatchNorm
#from torchmeta.modules.module import MetaModule

class _MetaBatchNorm(_BatchNorm, MetaModule):
    def forward(self, input, params=None):
        self._check_input_dim(input)
        if params is None:
            params = OrderedDict(self.named_parameters())
            
        # exponential_average_factor is self.momentum set to
        # (when it is available) only so that if gets updated
        # in ONNX graph when this node is exported to ONNX.
        if self.momentum is None:
            exponential_average_factor = 0.0
        else:
            exponential_average_factor = self.momentum

        if self.training and self.track_running_stats:
            if self.num_batches_tracked is not None:
                self.num_batches_tracked += 1
                if self.momentum is None:  # use cumulative moving average
                    exponential_average_factor = 1.0 / float(self.num_batches_tracked)
                else:  # use exponential moving average
                    exponential_average_factor = self.momentum

        weight = params.get('weight', None)
        bias = params.get('bias', None)

        return F.batch_norm(
            input, self.running_mean, self.running_var, weight, bias,
            self.training or not self.track_running_stats,
            exponential_average_factor, self.eps)

class MetaBatchNorm1d(_MetaBatchNorm):
    __doc__ = nn.BatchNorm1d.__doc__

    def _check_input_dim(self, input):
        if input.dim() != 2 and input.dim() != 3:
            raise ValueError('expected 2D or 3D input (got {}D input)'
                             .format(input.dim()))

class MetaBatchNorm2d(_MetaBatchNorm):
    __doc__ = nn.BatchNorm2d.__doc__

    def _check_input_dim(self, input):
        if input.dim() != 4:
            raise ValueError('expected 4D input (got {}D input)'
                             .format(input.dim()))

class MetaBatchNorm3d(_MetaBatchNorm):
    __doc__ = nn.BatchNorm3d.__doc__

    def _check_input_dim(self, input):
        if input.dim() != 5:
            raise ValueError('expected 5D input (got {}D input)'
                             .format(input.dim()))

In [6]:
import torch.nn as nn



class MetaSequential(nn.Sequential, MetaModule):
    __doc__ = nn.Sequential.__doc__

    def forward(self, input, params=None):
        for name, module in self._modules.items():
            if isinstance(module, MetaModule):
                input = module(input, params=get_subdict(params, name))
            elif isinstance(module, nn.Module):
                input = module(input)
            else:
                raise TypeError('The module must be either a torch module '
                    '(inheriting from `nn.Module`), or a `MetaModule`. '
                    'Got type: `{0}`'.format(type(module)))
        return input

In [7]:
!pip install torch_geometric
import torch.nn as nn
import torch.nn.functional as F

from collections import OrderedDict
from torch.nn.modules.utils import _single, _pair, _triple


class MetaConv1d(nn.Conv1d, MetaModule):
    __doc__ = nn.Conv1d.__doc__

    def forward(self, input, params=None):
        if params is None:
            params = OrderedDict(self.named_parameters())
        bias = params.get('bias', None)

        if self.padding_mode == 'circular':
            expanded_padding = ((self.padding[0] + 1) // 2, self.padding[0] // 2)
            return F.conv1d(F.pad(input, expanded_padding, mode='circular'),
                            params['weight'], bias, self.stride,
                            _single(0), self.dilation, self.groups)

        return F.conv1d(input, params['weight'], bias, self.stride,
                        self.padding, self.dilation, self.groups)

class MetaConv2d(nn.Conv2d, MetaModule):
    __doc__ = nn.Conv2d.__doc__

    def forward(self, input, params=None):
        if params is None:
            params = OrderedDict(self.named_parameters())
        bias = params.get('bias', None)

        if self.padding_mode == 'circular':
            expanded_padding = ((self.padding[1] + 1) // 2, self.padding[1] // 2,
                                (self.padding[0] + 1) // 2, self.padding[0] // 2)
            return F.conv2d(F.pad(input, expanded_padding, mode='circular'),
                            params['weight'], bias, self.stride,
                            _pair(0), self.dilation, self.groups)

        return F.conv2d(input, params['weight'], bias, self.stride,
                        self.padding, self.dilation, self.groups)

class MetaConv3d(nn.Conv3d, MetaModule):
    __doc__ = nn.Conv3d.__doc__

    def forward(self, input, params=None):
        if params is None:
            params = OrderedDict(self.named_parameters())
        bias = params.get('bias', None)

        if self.padding_mode == 'circular':
            expanded_padding = ((self.padding[2] + 1) // 2, self.padding[2] // 2,
                                (self.padding[1] + 1) // 2, self.padding[1] // 2,
                                (self.padding[0] + 1) // 2, self.padding[0] // 2)
            return F.conv3d(F.pad(input, expanded_padding, mode='circular'),
                            params['weight'], bias, self.stride,
                            _triple(0), self.dilation, self.groups)

        return F.conv3d(input, params['weight'], bias, self.stride,
                        self.padding, self.dilation, self.groups)

In [8]:
import torch
import torch.nn.init as init
import torch_geometric.nn as gnn

from collections import OrderedDict


class MetaGCNConv(gnn.GCNConv, MetaModule):
    __doc__ = gnn.GCNConv.__doc__
    
    def reset_parameters(self):
        init.xavier_uniform_(self.weight)
        init.zeros_(self.bias)
        self.cached_result = None
        self.cached_num_edges = None
        
    def forward(self, x, edge_index, edge_weight=None, params=None):
        self.normalize = False
        
        if params is None:
            params = OrderedDict(self.named_parameters())
        bias = params.get('bias', None)
        
        x = torch.matmul(x, params['weight'])
        
        if self.cached and self.cached_result is not None:
            if edge_index.size(1) != self.cached_num_edges:
                raise RuntimeError(
                    'Cached {} number of edges, but found {}. Please '
                    'disable the caching behavior of this layer by removing '
                    'the `cached=True` argument in its constructor.'.format(
                        self.cached_num_edges, edge_index.size(1)))

        if not self.cached or self.cached_result is None:
            self.cached_num_edges = edge_index.size(1)
            if self.normalize:
                edge_index, norm = self.norm(edge_index, x.size(self.node_dim),
                                             edge_weight, self.improved,
                                             x.dtype)
            else:
                norm = edge_weight
            self.cached_result = edge_index, norm
        
        edge_index, norm = self.cached_result

        return self.propagate(edge_index, x=x, norm=norm)

In [9]:
import math
import torch.nn as nn
import torch.nn.functional as F

from collections import OrderedDict

class MetaLinear(nn.Linear, MetaModule):
    __doc__ = nn.Linear.__doc__

    def reset_parameters(self):
        # nn.init.uniform_(self.weight, -0.1, 0.1)
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)
    
    def forward(self, input, params=None):
        if params is None:
            params = OrderedDict(self.named_parameters())
        bias = params.get('bias', None)
        return F.linear(input, params['weight'], bias)

class MetaBilinear(nn.Bilinear, MetaModule):
    __doc__ = nn.Bilinear.__doc__

    def forward(self, input1, input2, params=None):
        if params is None:
            params = OrderedDict(self.named_parameters())
        bias = params.get('bias', None)
        return F.bilinear(input1, input2, params['weight'], bias)

In [10]:
import torch
import torch.nn as nn

from collections import OrderedDict

class MetaModule(nn.Module):
    """
    Base class for PyTorch meta-learning modules. These modules accept an
    additional argument `params` in their `forward` method.

    Notes
    -----
    Objects inherited from `MetaModule` are fully compatible with PyTorch
    modules from `torch.nn.Module`. The argument `params` is a dictionary of
    tensors, with full support of the computation graph (for differentiation).
    """
    def meta_named_parameters(self, prefix='', recurse=True):
        gen = self._named_members(
            lambda module: module._parameters.items()
            if isinstance(module, MetaModule) else [],
            prefix=prefix, recurse=recurse)
        for elem in gen:
            yield elem

    def meta_parameters(self, recurse=True):
        for name, param in self.meta_named_parameters(recurse=recurse):
            yield param

In [11]:
import torch.nn as nn
import torch.nn.functional as F

from collections import OrderedDict

class MetaLayerNorm(nn.LayerNorm, MetaModule):
    __doc__ = nn.LayerNorm.__doc__

    def forward(self, input, params=None):
        if params is None:
            params = OrderedDict(self.named_parameters())
        weight = params.get('weight', None)
        bias = params.get('bias', None)
        return F.layer_norm(
            input, self.normalized_shape, weight, bias, self.eps)

In [12]:
import re
from collections import OrderedDict

def get_subdict(dictionary, key=None):
    if dictionary is None:
        return None
    if (key is None) or (key == ''):
        return dictionary
    key_re = re.compile(r'^{0}\.(.+)'.format(re.escape(key)))
    return OrderedDict((key_re.sub(r'\1', k), value) for (k, value)
        in dictionary.items() if key_re.match(k) is not None)

In [16]:
import os
import random
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import ViTModel

# -------------------------------
# BaseNet
# -------------------------------
class BaseNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        model_Res = models.resnet50(pretrained=True)
        self.resnet = nn.Sequential(*list(model_Res.children())[:-1])
        # Freeze everything except layer3, layer4
        for name, param in self.resnet.named_parameters():
            if not (name.startswith('layer3') or name.startswith('layer4')):
                param.requires_grad = False

        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        self.res_linear = nn.Linear(2048, 64)
        self.trans_linear = nn.Linear(768, 64)
        self.self_attention = nn.MultiheadAttention(embed_dim=64, num_heads=8)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, trans_b, res_b, params=None):
        device = trans_b.device
        # ResNet features
        res_features = self.resnet(res_b)
        res_features = torch.flatten(res_features, 1)
        if params is None:
            res_features = self.res_linear(res_features)
        else:
            w = params.get('res_linear.weight', self.res_linear.weight)
            b = params.get('res_linear.bias', self.res_linear.bias)
            res_features = F.linear(res_features, w.to(device), b.to(device))

        # ViT features
        vit_features = torch.mean(self.vit(trans_b).last_hidden_state, dim=1)
        if params is None:
            vit_features = self.trans_linear(vit_features)
        else:
            w = params.get('trans_linear.weight', self.trans_linear.weight)
            b = params.get('trans_linear.bias', self.trans_linear.bias)
            vit_features = F.linear(vit_features, w.to(device), b.to(device))

        # Self-attention fusion
        res_seq = res_features.unsqueeze(0)
        vit_seq = vit_features.unsqueeze(0)
        fused_seq, _ = self.self_attention(vit_seq, res_seq, vit_seq)
        fused = fused_seq.squeeze(0)

        if params is None:
            out = self.fc(fused)
        else:
            w = params.get('fc.weight', self.fc.weight)
            b = params.get('fc.bias', self.fc.bias)
            out = F.linear(fused, w.to(device), b.to(device))

        return out, fused

    def named_adaptable_parameters(self):
        return [
            ('res_linear.weight', self.res_linear.weight),
            ('res_linear.bias', self.res_linear.bias),
            ('trans_linear.weight', self.trans_linear.weight),
            ('trans_linear.bias', self.trans_linear.bias),
            ('fc.weight', self.fc.weight),
            ('fc.bias', self.fc.bias)
        ]

# -------------------------------
# CoLearner
# -------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, maxpool=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 3, padding=1),
                  nn.BatchNorm2d(out_ch), nn.ReLU()]
        if maxpool:
            layers.append(nn.MaxPool2d(2))
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)

class CoLearner(nn.Module):
    def __init__(self, in_channels, out_features, hidden_size):
        super().__init__()
        self.fixed_conv1 = ConvBlock(in_channels, hidden_size, maxpool=False)
        self.fixed_conv2 = ConvBlock(hidden_size, hidden_size)
        self.fixed_cls = None
        self.hidden_size = hidden_size
        self.out_features = out_features

    def forward(self, x):
        device = x.device
        features = self.fixed_conv1(x)
        features = self.fixed_conv2(features)
        features = features.view(features.size(0), -1)
        if self.fixed_cls is None:
            self.fixed_cls = nn.Linear(features.size(1), self.out_features).to(device)
        logits = self.fixed_cls(features)
        return features, logits

# -------------------------------
# Few-Shot Dataset
# -------------------------------
class FSLCDataset(Dataset):
    def __init__(self, folder, num_ways=5, num_shots=5, num_queries=5, transform=None):
        self.num_ways = num_ways
        self.num_shots = num_shots
        self.num_queries = num_queries
        self.transform = transform or transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor()
        ])
        self.data = {}
        for cls in sorted(os.listdir(folder)):
            cls_folder = os.path.join(folder, cls)
            if not os.path.isdir(cls_folder):
                continue
            imgs = [os.path.join(cls_folder,f) for f in os.listdir(cls_folder)
                    if f.lower().endswith(('.jpg','.JPG'))]
            if len(imgs) >= self.num_shots + self.num_queries:
                self.data[cls] = imgs
        self.classes = list(self.data.keys())
        if len(self.classes) < self.num_ways:
            raise ValueError(f"Not enough eligible classes: need {self.num_ways}, got {len(self.classes)}")

    def __len__(self):
        return 1000

    def __getitem__(self, idx):
        sampled_classes = random.sample(self.classes, self.num_ways)
        support_imgs, support_labels, query_imgs, query_labels = [], [], [], []
        for i, cls in enumerate(sampled_classes):
            imgs = random.sample(self.data[cls], self.num_shots + self.num_queries)
            support_imgs.extend(imgs[:self.num_shots])
            query_imgs.extend(imgs[self.num_shots:])
            support_labels.extend([i]*self.num_shots)
            query_labels.extend([i]*self.num_queries)

        support_tensors = [self.transform(Image.open(f).convert('RGB')) for f in support_imgs]
        query_tensors = [self.transform(Image.open(f).convert('RGB')) for f in query_imgs]
        return (torch.stack(support_tensors), torch.tensor(support_labels),
                torch.stack(query_tensors), torch.tensor(query_labels))

# -------------------------------
# Accuracy
# -------------------------------
def get_accuracy(logits, targets):
    _, preds = torch.max(logits, dim=1)
    return torch.mean((preds==targets).float())

# -------------------------------
# Helper for inner-loop param adaptation
# -------------------------------
def build_adaptable_param_dict(model, device):
    return {n: p.clone().detach().to(device).requires_grad_(True) for n,p in model.named_adaptable_parameters()}

def extract_adaptable_tensors(param_dict):
    keys = ['res_linear.weight','res_linear.bias','trans_linear.weight','trans_linear.bias','fc.weight','fc.bias']
    return [param_dict[k] for k in keys]

def update_param_dict_with_grads(param_dict, grads, lr_inner=0.5):
    keys = ['res_linear.weight','res_linear.bias','trans_linear.weight','trans_linear.bias','fc.weight','fc.bias']
    new = {}
    for k,g in zip(keys,grads):
        if g is None:
            new[k] = param_dict[k]
        else:
            new[k] = (param_dict[k] - lr_inner * g).requires_grad_(True)
    return new

# -------------------------------
# Training loop
# -------------------------------
def train_fsl_model(train_folder, val_folder, num_epochs=200, device='cuda:0',
                    inner_updates=1, meta_lr=1e-4, weight_decay=1e-5,
                    early_stop_patience=10, loss_scaling=1.0, episodes_per_epoch=10):

    device = torch.device(device if torch.cuda.is_available() else 'cpu')
    base_model = BaseNet(5).to(device)
    colearner = CoLearner(3,5,64).to(device)

    optimizer_base = torch.optim.Adam(base_model.parameters(), lr=meta_lr, weight_decay=weight_decay)
    optimizer_co = torch.optim.Adam(colearner.parameters(), lr=meta_lr, weight_decay=weight_decay)

    train_dataset = FSLCDataset(train_folder)
    val_dataset = FSLCDataset(val_folder)
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)

    best_val_acc = 0.0
    patience_counter = 0

    for epoch in range(num_epochs):
        base_model.train()
        colearner.train()

        ep_count = 0
        pbar = tqdm(train_loader, total=min(episodes_per_epoch,len(train_loader)),
                    desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
        for support_imgs, support_labels, query_imgs, query_labels in pbar:
            if ep_count >= episodes_per_epoch:
                break
            ep_count += 1
            support_imgs, support_labels = support_imgs.squeeze(0).to(device), support_labels.squeeze(0).to(device)
            query_imgs, query_labels = query_imgs.squeeze(0).to(device), query_labels.squeeze(0).to(device)

            # Inner-loop param adaptation
            param_dict = build_adaptable_param_dict(base_model, device)
            for _ in range(inner_updates):
                support_out,_ = base_model(support_imgs, support_imgs, params=param_dict)
                inner_loss = F.cross_entropy(support_out, support_labels)
                tensors = extract_adaptable_tensors(param_dict)
                grads = torch.autograd.grad(inner_loss, tensors, create_graph=True, allow_unused=True)
                grads = [g.to(device) if g is not None else torch.zeros_like(t) for g,t in zip(grads,tensors)]
                param_dict = update_param_dict_with_grads(param_dict, grads, lr_inner=0.5)

            # Outer-loop
            query_out,_ = base_model(query_imgs, query_imgs, params=param_dict)
            _, co_logits = colearner(query_imgs)
            loss_base = F.cross_entropy(query_out, query_labels)
            loss_co = F.cross_entropy(co_logits, query_labels)

            optimizer_base.zero_grad()
            optimizer_co.zero_grad()
            (loss_base + loss_scaling*loss_co).backward()
            optimizer_base.step()
            optimizer_co.step()

        # -------------------------------
        # Validation only
        # -------------------------------
        base_model.eval()
        colearner.eval()
        val_acc = 0
        vcount = 0
        with torch.no_grad():
            for support_imgs, support_labels, query_imgs, query_labels in val_loader:
                vcount += 1
                support_imgs, support_labels = support_imgs.squeeze(0).to(device), support_labels.squeeze(0).to(device)
                query_imgs, query_labels = query_imgs.squeeze(0).to(device), query_labels.squeeze(0).to(device)
                param_dict = build_adaptable_param_dict(base_model, device)
                query_out,_ = base_model(query_imgs, query_imgs, params=param_dict)
                val_acc += get_accuracy(query_out, query_labels).item()
        val_acc /= max(1,vcount)

        print(f"Epoch {epoch+1}: ValAcc = {val_acc:.4f}")
        if val_acc > best_val_acc + 1e-6:
            best_val_acc = val_acc
            print(f"-> New best val acc: {best_val_acc:.4f}")
            torch.save(base_model.state_dict(),"best_base_model.pt")
            torch.save(colearner.state_dict(),"best_colearner.pt")
        else:
            patience_counter += 1
            if patience_counter >= early_stop_patience:
                print("Early stopping triggered.")
                break

# -------------------------------
# Run
# -------------------------------
if __name__ == '__main__':
    train_folder = '/kaggle/input/fslcattledata/cattleFSL/UNEData'
    val_folder = '/kaggle/input/fslcattledata/cattleFSL/BeefCattle_Muzzle_Individualized'
    train_fsl_model(train_folder, val_folder,
                    num_epochs=200,
                    device='cuda:0',
                    inner_updates=1,
                    meta_lr=1e-4,
                    weight_decay=1e-5,
                    early_stop_patience=10,
                    loss_scaling=0.2,
                    episodes_per_epoch=1000)


Epoch 1: ValAcc = 0.2037
-> New best val acc: 0.2037
Epoch 2: ValAcc = 0.2080
-> New best val acc: 0.2080
Epoch 3: ValAcc = 0.2116
-> New best val acc: 0.2116
Epoch 4: ValAcc = 0.2156
-> New best val acc: 0.2156
Epoch 5: ValAcc = 0.2192
-> New best val acc: 0.2192
Epoch 6: ValAcc = 0.2226
-> New best val acc: 0.2226
Epoch 7: ValAcc = 0.2219
Epoch 8: ValAcc = 0.2219
Epoch 9: ValAcc = 0.2265
-> New best val acc: 0.2265
Epoch 10: ValAcc = 0.2308
-> New best val acc: 0.2308
Epoch 11: ValAcc = 0.2341
-> New best val acc: 0.2341
Epoch 12: ValAcc = 0.2378
-> New best val acc: 0.2378
Epoch 13: ValAcc = 0.2367
Epoch 14: ValAcc = 0.2363
Epoch 15: ValAcc = 0.2402
-> New best val acc: 0.2402
Epoch 16: ValAcc = 0.2445
-> New best val acc: 0.2445
Epoch 17: ValAcc = 0.2481
-> New best val acc: 0.2481
Epoch 18: ValAcc = 0.2517
-> New best val acc: 0.2517
Epoch 19: ValAcc = 0.2511
Epoch 20: ValAcc = 0.2549
-> New best val acc: 0.2549
Epoch 21: ValAcc = 0.2592
-> New best val acc: 0.2592
Epoch 22: ValAc

In [17]:
import os
import random
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import ViTModel

# -------------------------------
# BaseNet
# -------------------------------
class BaseNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        model_Res = models.resnet50(pretrained=True)
        self.resnet = nn.Sequential(*list(model_Res.children())[:-1])
        # Freeze everything except layer3, layer4
        for name, param in self.resnet.named_parameters():
            if not (name.startswith('layer3') or name.startswith('layer4')):
                param.requires_grad = False

        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        self.res_linear = nn.Linear(2048, 64)
        self.trans_linear = nn.Linear(768, 64)
        self.self_attention = nn.MultiheadAttention(embed_dim=64, num_heads=8)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, trans_b, res_b, params=None):
        device = trans_b.device
        # ResNet features
        res_features = self.resnet(res_b)
        res_features = torch.flatten(res_features, 1)
        if params is None:
            res_features = self.res_linear(res_features)
        else:
            w = params.get('res_linear.weight', self.res_linear.weight)
            b = params.get('res_linear.bias', self.res_linear.bias)
            res_features = F.linear(res_features, w.to(device), b.to(device))

        # ViT features
        vit_features = torch.mean(self.vit(trans_b).last_hidden_state, dim=1)
        if params is None:
            vit_features = self.trans_linear(vit_features)
        else:
            w = params.get('trans_linear.weight', self.trans_linear.weight)
            b = params.get('trans_linear.bias', self.trans_linear.bias)
            vit_features = F.linear(vit_features, w.to(device), b.to(device))

        # Self-attention fusion
        res_seq = res_features.unsqueeze(0)
        vit_seq = vit_features.unsqueeze(0)
        fused_seq, _ = self.self_attention(vit_seq, res_seq, vit_seq)
        fused = fused_seq.squeeze(0)

        if params is None:
            out = self.fc(fused)
        else:
            w = params.get('fc.weight', self.fc.weight)
            b = params.get('fc.bias', self.fc.bias)
            out = F.linear(fused, w.to(device), b.to(device))

        return out, fused

    def named_adaptable_parameters(self):
        return [
            ('res_linear.weight', self.res_linear.weight),
            ('res_linear.bias', self.res_linear.bias),
            ('trans_linear.weight', self.trans_linear.weight),
            ('trans_linear.bias', self.trans_linear.bias),
            ('fc.weight', self.fc.weight),
            ('fc.bias', self.fc.bias)
        ]

# -------------------------------
# CoLearner (Updated Architecture)
# -------------------------------
class CoLearner(nn.Module):
    def __init__(self, in_channels, out_features, hidden_size):
        super().__init__()
        # Two convolutional layers
        self.conv1 = nn.Conv2d(in_channels, hidden_size, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(hidden_size)
        self.relu1 = nn.ReLU()

        self.conv2 = nn.Conv2d(hidden_size, hidden_size, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(hidden_size)
        self.relu2 = nn.ReLU()

        # Adaptive average pooling
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1,1))

        # Two fully connected layers
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.relu_fc = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, out_features)

    def forward(self, x):
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        x = self.relu_fc(self.fc1(x))
        logits = self.fc2(x)
        return x, logits

# -------------------------------
# Few-Shot Dataset
# -------------------------------
class FSLCDataset(Dataset):
    def __init__(self, folder, num_ways=5, num_shots=5, num_queries=5, transform=None):
        self.num_ways = num_ways
        self.num_shots = num_shots
        self.num_queries = num_queries
        self.transform = transform or transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor()
        ])
        self.data = {}
        for cls in sorted(os.listdir(folder)):
            cls_folder = os.path.join(folder, cls)
            if not os.path.isdir(cls_folder):
                continue
            imgs = [os.path.join(cls_folder,f) for f in os.listdir(cls_folder)
                    if f.lower().endswith(('.jpg','.JPG'))]
            if len(imgs) >= self.num_shots + self.num_queries:
                self.data[cls] = imgs
        self.classes = list(self.data.keys())
        if len(self.classes) < self.num_ways:
            raise ValueError(f"Not enough eligible classes: need {self.num_ways}, got {len(self.classes)}")

    def __len__(self):
        return 1000

    def __getitem__(self, idx):
        sampled_classes = random.sample(self.classes, self.num_ways)
        support_imgs, support_labels, query_imgs, query_labels = [], [], [], []
        for i, cls in enumerate(sampled_classes):
            imgs = random.sample(self.data[cls], self.num_shots + self.num_queries)
            support_imgs.extend(imgs[:self.num_shots])
            query_imgs.extend(imgs[self.num_shots:])
            support_labels.extend([i]*self.num_shots)
            query_labels.extend([i]*self.num_queries)

        support_tensors = [self.transform(Image.open(f).convert('RGB')) for f in support_imgs]
        query_tensors = [self.transform(Image.open(f).convert('RGB')) for f in query_imgs]
        return (torch.stack(support_tensors), torch.tensor(support_labels),
                torch.stack(query_tensors), torch.tensor(query_labels))

# -------------------------------
# Accuracy
# -------------------------------
def get_accuracy(logits, targets):
    _, preds = torch.max(logits, dim=1)
    return torch.mean((preds==targets).float())

# -------------------------------
# Helper for inner-loop param adaptation
# -------------------------------
def build_adaptable_param_dict(model, device):
    return {n: p.clone().detach().to(device).requires_grad_(True) for n,p in model.named_adaptable_parameters()}

def extract_adaptable_tensors(param_dict):
    keys = ['res_linear.weight','res_linear.bias','trans_linear.weight','trans_linear.bias','fc.weight','fc.bias']
    return [param_dict[k] for k in keys]

def update_param_dict_with_grads(param_dict, grads, lr_inner=0.5):
    keys = ['res_linear.weight','res_linear.bias','trans_linear.weight','trans_linear.bias','fc.weight','fc.bias']
    new = {}
    for k,g in zip(keys,grads):
        if g is None:
            new[k] = param_dict[k]
        else:
            new[k] = (param_dict[k] - lr_inner * g).requires_grad_(True)
    return new

# -------------------------------
# Training loop
# -------------------------------
def train_fsl_model(train_folder, val_folder, num_epochs=200, device='cuda:0',
                    inner_updates=1, meta_lr=1e-4, weight_decay=1e-5,
                    early_stop_patience=10, loss_scaling=1.0, episodes_per_epoch=10):

    device = torch.device(device if torch.cuda.is_available() else 'cpu')
    base_model = BaseNet(5).to(device)
    colearner = CoLearner(3,5,64).to(device)

    optimizer_base = torch.optim.Adam(base_model.parameters(), lr=meta_lr, weight_decay=weight_decay)
    optimizer_co = torch.optim.Adam(colearner.parameters(), lr=meta_lr, weight_decay=weight_decay)

    train_dataset = FSLCDataset(train_folder)
    val_dataset = FSLCDataset(val_folder)
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)

    best_val_acc = 0.0
    patience_counter = 0

    for epoch in range(num_epochs):
        base_model.train()
        colearner.train()

        ep_count = 0
        pbar = tqdm(train_loader, total=min(episodes_per_epoch,len(train_loader)),
                    desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
        for support_imgs, support_labels, query_imgs, query_labels in pbar:
            if ep_count >= episodes_per_epoch:
                break
            ep_count += 1
            support_imgs, support_labels = support_imgs.squeeze(0).to(device), support_labels.squeeze(0).to(device)
            query_imgs, query_labels = query_imgs.squeeze(0).to(device), query_labels.squeeze(0).to(device)

            # Inner-loop param adaptation
            param_dict = build_adaptable_param_dict(base_model, device)
            for _ in range(inner_updates):
                support_out,_ = base_model(support_imgs, support_imgs, params=param_dict)
                inner_loss = F.cross_entropy(support_out, support_labels)
                tensors = extract_adaptable_tensors(param_dict)
                grads = torch.autograd.grad(inner_loss, tensors, create_graph=True, allow_unused=True)
                grads = [g.to(device) if g is not None else torch.zeros_like(t) for g,t in zip(grads,tensors)]
                param_dict = update_param_dict_with_grads(param_dict, grads, lr_inner=0.5)

            # Outer-loop
            query_out,_ = base_model(query_imgs, query_imgs, params=param_dict)
            _, co_logits = colearner(query_imgs)
            loss_base = F.cross_entropy(query_out, query_labels)
            loss_co = F.cross_entropy(co_logits, query_labels)

            optimizer_base.zero_grad()
            optimizer_co.zero_grad()
            (loss_base + loss_scaling*loss_co).backward()
            optimizer_base.step()
            optimizer_co.step()

        # -------------------------------
        # Validation only
        # -------------------------------
        base_model.eval()
        colearner.eval()
        val_acc = 0
        vcount = 0
        with torch.no_grad():
            for support_imgs, support_labels, query_imgs, query_labels in val_loader:
                vcount += 1
                support_imgs, support_labels = support_imgs.squeeze(0).to(device), support_labels.squeeze(0).to(device)
                query_imgs, query_labels = query_imgs.squeeze(0).to(device), query_labels.squeeze(0).to(device)
                param_dict = build_adaptable_param_dict(base_model, device)
                query_out,_ = base_model(query_imgs, query_imgs, params=param_dict)
                val_acc += get_accuracy(query_out, query_labels).item()
        val_acc /= max(1,vcount)

        print(f"Epoch {epoch+1}: ValAcc = {val_acc:.4f}")
        if val_acc > best_val_acc + 1e-6:
            best_val_acc = val_acc
            print(f"-> New best val acc: {best_val_acc:.4f}")
            torch.save(base_model.state_dict(),"best_base_model.pt")
            torch.save(colearner.state_dict(),"best_colearner.pt")
        else:
            patience_counter += 1
            if patience_counter >= early_stop_patience:
                print("Early stopping triggered.")
                break

# -------------------------------
# Run
# -------------------------------
if __name__ == '__main__':
    train_folder = '/kaggle/input/fslcattledata/cattleFSL/UNEData'
    val_folder = '/kaggle/input/fslcattledata/cattleFSL/BeefCattle_Muzzle_Individualized'
    train_fsl_model(train_folder, val_folder,
                    num_epochs=200,
                    device='cuda:0',
                    inner_updates=1,
                    meta_lr=1e-4,
                    weight_decay=1e-5,
                    early_stop_patience=10,
                    loss_scaling=0.2,
                    episodes_per_epoch=1000)


Epoch 1: ValAcc = 0.1968
-> New best val acc: 0.1968
Epoch 2: ValAcc = 0.2138
-> New best val acc: 0.2138
Epoch 3: ValAcc = 0.2282
-> New best val acc: 0.2282
Epoch 4: ValAcc = 0.2416
-> New best val acc: 0.2416
Epoch 5: ValAcc = 0.2578
-> New best val acc: 0.2578
Epoch 6: ValAcc = 0.2415
Epoch 7: ValAcc = 0.2559
Epoch 8: ValAcc = 0.2705
-> New best val acc: 0.2705
Epoch 9: ValAcc = 0.2540
Epoch 10: ValAcc = 0.2625
Epoch 11: ValAcc = 0.2810
-> New best val acc: 0.2810
Epoch 12: ValAcc = 0.2961
-> New best val acc: 0.2961
Epoch 13: ValAcc = 0.3094
-> New best val acc: 0.3094
Epoch 14: ValAcc = 0.3273
-> New best val acc: 0.3273
Epoch 15: ValAcc = 0.3205
Epoch 16: ValAcc = 0.3045
Epoch 17: ValAcc = 0.3118
Epoch 18: ValAcc = 0.3305
-> New best val acc: 0.3305
Epoch 19: ValAcc = 0.3479
-> New best val acc: 0.3479
Epoch 20: ValAcc = 0.3579
-> New best val acc: 0.3579
Epoch 21: ValAcc = 0.3757
-> New best val acc: 0.3757
Epoch 22: ValAcc = 0.3883
-> New best val acc: 0.3883
Epoch 23: ValAcc 